# Word-aligned EEG + text sentiment with a LoRA LLM

Each word of the sentence is followed by one **EEG token**: the reader's fixation-locked EEG for that word (ZuCo band power over the word's total reading time, 8 bands × 104 electrodes, standardized per reader), mapped into the LLM's embedding space by a small trained projector. A LoRA-tuned **Qwen2.5-1.5B-Instruct** reads the whole sequence and answers negative / neutral / positive.

Four arms are trained identically (same initial weights, batch order, folds); only the EEG input differs:

| arm | input |
|---|---|
| `text_only` | sentence only |
| `text_eeg` | sentence + the reader's EEG after each word |
| `text_shuffled_eeg` | same, EEG vectors re-assigned among the reader's words (alignment destroyed) |
| `text_fixation_only` | EEG tokens only say whether the word was fixated |

**The model uses EEG only if `text_eeg` beats `text_shuffled_eeg`** (paired sentence-cluster CI above 0, p < 0.05). Beating `text_only` alone is not enough. Unseen-sentence protocol, 5 folds. Every fold is saved as it finishes, so a disconnect loses at most one fold. Use a GPU runtime.

## 1. Repository and dependencies

In [ ]:
import os
REPO_URL = 'https://github.com/MohammadJRanjbar/zuco-multimodal-sentiment.git'
BRANCH = 'feature/neurolm-eeg-probe'
REPO_DIR = '/content/zuco-multimodal-sentiment'
if not os.path.exists(REPO_DIR):
    !git clone --branch $BRANCH $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR fetch origin && git -C $REPO_DIR checkout $BRANCH && git -C $REPO_DIR pull --ff-only
%cd $REPO_DIR
!pip install -q -r requirements-neurolm.txt
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Drive and paths

The Hugging Face cache lives on Drive, so the LLM downloads once (about 3 GB for the 1.5B model).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
THESIS_ROOT = '/content/drive/MyDrive/Thesis'
MAT_DIR = f'{THESIS_ROOT}/Data/zuco_og_raw'
LABELS_CSV = f'{THESIS_ROOT}/Data/zuco_sentiment_labels_task1_fixed.csv'
ARTIFACTS = f'{THESIS_ROOT}/CachedArtifacts/zuco_multimodal_sentiment'
WORD_EEG_DIR = f'{ARTIFACTS}/word_eeg/TRT'
RESULTS_BASE = f'{THESIS_ROOT}/Results/zuco_multimodal_sentiment/eeg_text_lora'
RUN_TAG = 'v1_word_aligned'
os.environ['HF_HOME'] = f'{ARTIFACTS}/hf_cache'
for path in [MAT_DIR, LABELS_CSV]:
    assert os.path.exists(path), f'missing: {path}'
os.makedirs(RESULTS_BASE, exist_ok=True)

# Faster repeated reads: copy the .mat files to local disk once per runtime.
LOCAL_MAT = '/content/zuco_og_raw'
!mkdir -p $LOCAL_MAT && rsync -a --include='results*_SR.mat' --exclude='*' "$MAT_DIR/" "$LOCAL_MAT/"
MAT_DIR = LOCAL_MAT

## 3. Tests

Includes a check that the model *can* learn an EEG-only signal when one exists, while the shuffled and text-only arms cannot.

In [ ]:
!python -m pytest -q tests/test_fusion.py tests/test_word_eeg.py

## 4. Extract word-level EEG (once; resumable)

In [ ]:
!python scripts/extract_word_eeg.py --mat-dir "$MAT_DIR" --labels-csv "$LABELS_CSV" --out-dir "$WORD_EEG_DIR"
import json
print(json.dumps(json.load(open(f'{WORD_EEG_DIR}/word_eeg_summary.json')), indent=2))

## 5. Quick run (small model, 1 fold, 1 epoch, text only vs text + EEG)

Checks the full path on the GPU in a few minutes. Not a result.

In [ ]:
!python scripts/run_eeg_text_lora.py --word-eeg-dir "$WORD_EEG_DIR" --results-dir "$RESULTS_BASE" --run-tag {RUN_TAG}_quick --quick

## 6. Full run: 4 arms × 5 folds with Qwen2.5-1.5B

Resumable: re-run this cell after a disconnect. On a T4 add `--base-dtype float32` if the loss turns non-finite.

In [ ]:
!python scripts/run_eeg_text_lora.py --word-eeg-dir "$WORD_EEG_DIR" --results-dir "$RESULTS_BASE" --run-tag $RUN_TAG

## 7. Report

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f'{RESULTS_BASE}/{RUN_TAG}/eeg_text_lora_results.md').read()))